In [1]:
# Setup: Add project root to Python path
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")


Project root: /Users/ilyasyeskenov/Desktop/req_check


In [2]:
#The total flow rate for 3 hydrant outlets shall be 1,350 l/min.

In [2]:
# Import required modules
from clients.supabase_client import SupabaseClient
from config.config import TOP_K_CHUNKS

# Initialize Supabase client
supabase_client = SupabaseClient()
print("Supabase client initialized successfully!")

Supabase client initialized successfully!


In [3]:
from clients.supabase_client import SupabaseClient

# Initialize your custom client
supabase_client = SupabaseClient()

# Access the underlying Supabase client via .client property
response = supabase_client.client.from_('Project').select('id, name').execute()
projects = response.data

# Find project by name
landscape_project = next((p for p in projects if 'landscape' in p['name'].lower()), None)
project_id = landscape_project['id'] if landscape_project else None

In [4]:
supabase_client.search_chunks_by_project_id

<bound method SupabaseClient.search_chunks_by_project_id of <clients.supabase_client.SupabaseClient object at 0x10cb89750>>

In [5]:
project_id = "93d3a25b-d15d-4689-affb-d027bbc422e7"

### Text search only

In [6]:
from text_search import TextSearchBackend

In [11]:
# Minimal test for TextSearchBackend
backend = TextSearchBackend()
query = "'Fire Hydrant/Hose Reel (FHR)'"
results = backend.get_chunks_by_text_search_and_project_id(query, project_id, limit=3)

print(f"Found {len(results)} chunks")
for i, chunk in enumerate(results, 1):
    print(f"\n{i}. File: {chunk.get('file_name', 'N/A')}")
    print(f"   Content: {chunk.get('content', '')[:150]}...")

Found 1 chunks

1. File: N/A
   Content: ## 3.3.2 Design Standards

(a) The Fire Services Systems design will be carried out in accordance with the following Standards and References:

. MTR ...


[{'id': '6805376a-f252-46fd-bfd4-e17ec25ec6c2',
  'content': '## 3.3.2 Design Standards\n\n(a) The Fire Services Systems design will be carried out in accordance with the following Standards and References:\n\n. MTR - Design Standards Manual for New Works and Major Asset Replacement Works (NW&MARWDSM) - Sections 7 Electrical and Mechanical Systems (Revision A2)\n\n· MTR - Materials and Workmanship Specifications for Building Services\n\n· Loss Prevention Council (LPC) Rules for Automatic Sprinkler Installations incorporating BS EN 12845: 2015 - Fixed Fire fighting Systems - Automatic Sprinkler Systems - Design, Installation and Maintenance and the Technical Guidance with suitable modification pertinent to Hong Kong issued by the Fire Services Department (FSD), Hong Kong\n\n· BS 5839-1:2017 (Incorporating Corrigendum No.1) Fire Detection and Fire Alarm Systems for Buildings - Part 1: Code of Practice for Design, Installation, Commissioning and Maintenance of Systems in Non-Domestic Prem

### Hybrid Search

In [7]:
from text_search import TextSearchBackend

def get_hybrid_chunks(query, project_id, vector_k=6, text_k=4):
    """Combines vector (semantic) and text (keyword) search for better RAG retrieval."""
    try:
        # 1. Semantic Search (Vector)
        query_embedding = supabase_client.ai_client.generate_embedding(query)
        vector_chunks = supabase_client.get_chunks_by_project_id(
            project_id=project_id,
            query_embedding=query_embedding,
            match_count=vector_k
        )
        
        # 2. Keyword Search (Text) - Use TextSearchBackend
        backend = TextSearchBackend()
        text_chunks = backend.get_chunks_by_text_search_and_project_id(
            query=query,
            project_id=project_id,
            limit=text_k
        )
        
        # 3. Merge
        combined_chunks = merge_search_results(vector_chunks, text_chunks)
        
        # Format for output
        merged_text = "\n".join(
            f"chunk-{i+1}: {chunk.get('content', '')}"
            for i, chunk in enumerate(combined_chunks)
        )
        return merged_text, combined_chunks
        
    except Exception as e:
        print(f"❌ Hybrid search error: {str(e)}")
        return "", []

def merge_search_results(vector_results, text_results):
    """Merge all vector and text search results, removing duplicates (no priority)."""
    seen_ids = set()
    merged = []
    all_results = vector_results + text_results
    for result in all_results:
        result_id = result.get('id') or result.get('chunk_id')
        if result_id and result_id not in seen_ids:
            merged.append(result)
            seen_ids.add(result_id)
    return merged

In [11]:
# 2. Keyword Search (Text) - Use TextSearchBackend
backend = TextSearchBackend()

In [ ]:
print("""Area: Cable Termination Rm (For Comm. System)
 F.S. Systems Detection: PS Fire Hydrant/ 
 Hose Reel Coverage: Y 
 Automatic Fire Suppression System: N 
 Portable Extinguishers: Y"""

'Area: Cable Termination Rm (For Comm. System) F.S. Systems Detection: PS Fire Hydrant/ Hose Reel Coverage: Y Automatic Fire Suppression System: N Portable Extinguishers: Y'


In [24]:
query_full = "'Area: Cable Termination Rm (For Comm. System) F.S. Systems Detection: PS Fire Hydrant/ Hose Reel Coverage: Y Automatic Fire Suppression System: N Portable Extinguishers: Y'"
query2 = "'Fire Hydrant/Hose Reel (FHR)'"

In [21]:
# 1. Semantic Search (Vector)
query_embedding = supabase_client.ai_client.generate_embedding(query2)
vector_chunks = supabase_client.get_chunks_by_project_id(
    project_id=project_id,
    query_embedding=query_embedding,
    match_count=5
)

text_chunks = backend.get_chunks_by_text_search_and_project_id(
    query=query2,
    project_id=project_id,
    limit=5
)

# 3. Merge
combined_chunks = merge_search_results(vector_chunks, text_chunks)

In [26]:
vector_chunks

[{'id': '5c248443-dbc7-4589-b3f5-95f44a2866e0',
  'content': '## 15.3 Fire Services (FS)\n\n## 15.3.1 Design Criteria and Parameters\n\n(a) The Fire Services Provisions for the various rooms and locations at Feeder Station are as indicated in Table 15.3.1 below:\n\nTable 15.3.1: Fire Services Provisions for Feeder Station\n\n<table>\n  <thead>\n    <tr>\n      <th rowspan="2">Room/Facility</th>\n      <th colspan="5">FS Systems</th>\n    </tr>\n    <tr>\n      <th>Detection</th>\n      <th>Fire Hydrant / Hose Reel Coverage</th>\n      <th>Automatic Suppression Systems</th>\n      <th>Portable Extinguishers</th>\n      <th>Water Spray System</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>CLP L.V. Switch Room</td>\n      <td>PS</td>\n      <td>Y</td>\n      <td>N</td>\n      <td>Y</td>\n      <td>N</td>\n    </tr>\n    <tr>\n      <td>CLP Protection / Control Room</td>\n      <td>PS</td>\n      <td>Y</td>\n      <td>N</td>\n      <td>Y</td>\n      <td>N</td>\n    </tr>\n    

In [27]:
text_chunks

[]

In [25]:
# 1. Semantic Search (Vector)
query_embedding = supabase_client.ai_client.generate_embedding(query_full)

vector_chunks = supabase_client.get_chunks_by_project_id(
    project_id=project_id,
    query_embedding=query_embedding,
    match_count=5
)

text_chunks = backend.get_chunks_by_text_search_and_project_id(
    query=query_full,
    project_id=project_id,
    limit=5
)

# 3. Merge
combined_chunks = merge_search_results(vector_chunks, text_chunks)

#### Testing


In [21]:
# Edit prompt template for testing with query and combined_chunks as supporting docs

json_example = """
{
  "status": "FULFILLED" | "PARTIALLY_FULFILLED" | "NOT_FULFILLED",
  "relevance_score": <integer_0_to_10>,
  "justification": "<A detailed explanation of how the document satisfies the requirement. If partially fulfilled, explicitly explain what is missing.>",
  "citations": [
    {
      "source_text": "<The exact verbatim quote from the document used as evidence>",
      "document_reference": "<The name, page number, or ID of the specific document chunk if available>"
    }
  ]
}
"""

test_prompt = (
    "You are an expert Construction Compliance Auditor and Quality Assurance Specialist. "
    "Your role is to rigorously analyze technical documentation to verify if specific project requirements have been met.\n\n"
    "**Your Goal:**\n"
    "Analyze the provided supporting documents to determine if the specific Requirement (provided below) is fulfilled. But first rank the document according to the query you have received. "
    "You must provide a justification based *strictly* on the evidence found in the text.\n\n"
    "**Input Context:**\n"
    f"- **Requirement to Verify:** \"{query}\"\n"
    "- **Supporting Documents:**\n"
    + "\n".join([
        f"---\n[File: {chunk.get('file_name', 'N/A')}] (page {chunk.get('page_number', 'N/A')}):\n{chunk.get('content', '')[:750]}"
        for chunk in combined_chunks
      ])
    + "\n\n"
    "**Step-by-Step Reasoning Process:**\n"
    "1.  **Analyze the Requirement:** Break down the specific requirement into its constituent conditions (e.g., specific materials, dimensions, safety standards, certifications, or tolerances).\n"
    "2.  **Scan for Evidence:** Search the Supporting Documents for exact keywords, synonyms, or technical specifications that match the requirement's conditions.\n"
    "3.  **Evaluate Completeness:** Determine if the evidence covers the *entirety* of the requirement or only parts of it.\n"
    "4.  **Formulate Justification:** Construct an argument linking the text in the documents to the requirement conditions.\n\n"
    "**Output Instructions:**\n"
    "Provide your response in a valid JSON format with the following structure (do NOT include extra markdown code blocks):\n\n"
    f"{json_example}\n"
)

In [22]:
from openai import OpenAI
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env file from parent directory (req_check folder)
env_path = Path().resolve().parent / '.env'
load_dotenv(env_path)

# Get API key from environment
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file. Please set it in the parent directory's .env file.")

# Initialize OpenAI client (v1.0+ API)
client = OpenAI(api_key=api_key)
gpt_model = "gpt-4o"

def get_gpt4o_completion(prompt, system_message=None, temperature=0.2, max_tokens=800):
    """
    Get GPT-4o completion using OpenAI v1.0+ API.
    
    Args:
        prompt: User prompt/query
        system_message: Optional system message
        temperature: Temperature for generation (default: 0.2)
        max_tokens: Maximum tokens in response (default: 800)
    
    Returns:
        Generated text response
    """
    messages = []
    if system_message:
        messages.append({"role": "system", "content": system_message})
    messages.append({"role": "user", "content": prompt})
    
    # Use new OpenAI v1.0+ API
    response = client.chat.completions.create(
        model=gpt_model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    
    return response.choices[0].message.content

In [23]:
# Example usage (to actually call, remove comments):
ai_output = get_gpt4o_completion(test_prompt)
print(ai_output)


{
  "status": "FULFILLED",
  "relevance_score": 10,
  "justification": "The requirement for a 'Fire Hydrant/Hose Reel (FHR)' system is comprehensively addressed across multiple documents. The system is described in detail, including the provision of a 36m3 water storage tank, fixed fire pumps, and a jockey pump, with specifications for water pressure and flow rate. Additionally, the system's installation is aligned with the FSI Code, ensuring compliance with relevant standards. The documents also specify the positioning of inlets and the reach of the hose reel tubing, confirming that all necessary components and standards for the FHR system are met.",
  "citations": [
    {
      "source_text": "Fire hydrant and hose reel system (FH/HR) will be provided in feeder station. The system will comprise a 36m3 FS reinforced concrete water storage tank, two number fixed fire pumps (duty/standby), a jockey pump and associated pipework to distribute water to the FH/HR installations.",
      "doc

In [ ]:
"""Break down the provided requirement into individual sub-requirements if it is complex.
A requirement is considered complex if it involves multiple actions, systems, equipment, or processes.
If the requirement is simple (e.g., involves a single action or system), return the original requirement as is.

Instructions:
Generate up to 5 sub-requirements, ensuring each is distinct and relevant to the original requirement.
Each sub-requirement must be self-contained and include all key systems, equipment, locations, and processes mentioned in the original requirement, even if this causes repetition.
Exclude references to actors (e.g., “the Contractor,” “the operator,” “staff”). Only include systems, equipment, locations, and processes.
Do not reword the original requirement excessively; preserve its terminology and original technical meaning.
If the requirement does not specify systems, equipment, locations, or processes, note this clearly and proceed with the breakdown based on available information.

Example Requirement:
(S320.2B.16.1) (c) All SAMS alarms can be monitored through the MCS workstation. The geographic location of an activated alarm is clearly shown on a station layout display.

Sub-requirements Generated:
All SAMS alarms can be monitored through the MCS workstation.
The geographic location of an activated SAMS alarm is clearly shown on a station layout display.

Example Complex Requirement:
(S450.1A.22.3) The HVAC system must maintain temperature control within ±2°C and log performance data to the BMS, while the control panel displays real-time status updates.

Sub-requirements Generated:
The HVAC system maintains temperature control within ±2°C.
The HVAC system logs performance data to the BMS.
The HVAC system control panel displays real-time status updates of the HVAC system within the BMS environment.

User Query: {{query}}"""